# **Agent's tools**

## **Imports and environment configs**

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from openai import OpenAI

MODEL = "gpt-5.4-mini"

openai_client = OpenAI()

In [3]:
import json

from bibliometric_rag_assistant.ingestion.pubmed import search_pubmed
from bibliometric_rag_assistant.search_strategy import PUBMED_QUERY

## **Simple Agent - answers without tools**

In [4]:
history = [
    {"role": "user", "content": "What is an agent, properly?"}
]


response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=history
)

print(response.output_text)

An **agent**, properly speaking, is **something that acts on its own initiative in order to bring about outcomes**.

More formally, an agent is an entity that:

1. **Perceives** something about its environment,
2. **Forms or selects goals**,
3. **Chooses actions**,
4. **Acts in a way that tends to change the world** toward those goals.

### Core idea
An agent is not just a thing that exists; it is a thing that **does**.

### Common ingredients of agency
- **Autonomy**: it can initiate action rather than merely being pushed around.
- **Goal-directedness**: its actions are organized around some end.
- **Responsiveness**: it adjusts to circumstances and feedback.
- **Control or influence**: its actions make a difference.

### Examples
- A **person** deciding to call a friend.
- A **robot vacuum** navigating a room.
- A **software agent** booking travel.
- In philosophy, sometimes even an **animal** counts as an agent; in some discussions, a corporation can too.

### Important distinction


In [5]:
history += response.output

history.append({"role": "user", "content": "What specifically is an AI agent?"})

second_response = openai_client.responses.create(
    model=MODEL,
    input=history,
)

print(second_response.output_text)

An **AI agent** is a system that **perceives inputs, decides what to do, and acts to achieve a goal**.

More specifically, it usually has these features:

1. **Inputs from an environment**  
   It gets observations: text, sensor data, API responses, images, state changes, etc.

2. **Decision-making**  
   It selects actions based on the input and some objective, policy, or instructions.

3. **Action execution**  
   It can do things that affect the world: generate text, call tools, move a robot, send an email, place trades, etc.

4. **Goal-directed behavior**  
   Its actions are organized around a target outcome, not just producing a one-off response.

### A simple contrast
- A **chatbot** answers a question and stops.
- An **AI agent** may answer, then decide to search the web, use a tool, check results, revise its plan, and continue until the goal is met.

### Typical AI-agent loop
**Observe → think/plan → act → observe results → repeat**

### Important nuance
Not every AI system is

In [6]:
history += second_response.output

history.append({"role": "user", "content": "According with the last answers, will you become an agent when i provide tools and call you in a loop?"})

third_response = openai_client.responses.create(
    model=MODEL,
    input=history,
)

print(third_response.output_text)

Yes — **if** you wrap me in a loop, give me tools, and let me choose and execute actions toward a goal, then the resulting system can be described as an **AI agent**.

But there’s an important distinction:

### Me alone vs. me as part of a system
- **Me alone**: I’m a model that generates outputs from inputs.
- **Me in a tool-using loop**: I can function as the “decision-making” component of an agent.

So the **agent** is not just the model; it’s the **whole setup**:
- the model,
- the tools,
- the loop/controller,
- the memory/state,
- and the goal or policy.

### What changes when you add a loop?
If you call me repeatedly and each turn I can:
- inspect prior results,
- decide whether to search, compute, write, or stop,
- use tools,
- adapt based on feedback,

then the overall system exhibits **agency**.

### But am I “really” an agent?
That depends on how strictly you use the term.

- In a **practical engineering sense**: yes, that setup is an agentic system.
- In a **strict philosop

## **Defining a tool**

In [7]:
search_pubmed_tool = {
    "type": "function",
    "name": "search_pubmed",
    "description": (
        "Search PubMed and return the total number of matching records "
        "and a limited list of PMIDs. Use this for PubMed literature searches."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": (
                    "PubMed query using its Boolean operators and field tags, "
                    "such as [Title/Abstract] and [MeSH Terms]."
                ),
            },
            "limit": {
                "type": "integer",
                "minimum": 1,
                "maximum": 20,
                "description": (
                    "Maximum number of PMIDs to return. "
                    "Use 20 when the user does not specify a limit."
                ),
            },
        },
        "required": ["query", "limit"],
        "additionalProperties": False,
    },
    "strict": True,
}


## **Sending the question with the tool**

Mesmo com a ferramenta, vai responder "na lata". Não precisa de decidir se chama o não a ferramenta, já que a pergunta não tem a menor relação com o que a ferramenta pode fazer. 

In [8]:
history = [
    {"role": "user", "content": "What is an agent, properly?"}
]


response = openai_client.responses.create(
    model=MODEL,
    input=history,
    tools=[search_pubmed_tool]  # SCHEMA/JSON definition
)

for item in response.output:
    print(item.type, item)

message ResponseOutputMessage(id='msg_0231fe845957a4b5006ac92e34961487d2a4f69385b768f12a', content=[ResponseOutputText(annotations=[], text='“Agent” can mean different things depending on the field, but in the most general sense:\n\n**An agent is something that acts on behalf of a goal, and whose actions are responsive to its environment.**\n\nA good, broad definition usually includes these pieces:\n\n1. **It perceives or observes** something about the world.\n2. **It chooses actions** based on what it perceives.\n3. **Its actions affect outcomes** in the world.\n4. **It is organized around some objective, preference, or function**—even if only implicitly.\n\n### In philosophy and everyday language\nAn agent is a being that can **intend, decide, and act**.  \nA person is a paradigm agent because they can deliberate and choose among alternatives.\n\n### In economics and social science\nAn agent is an entity that **makes decisions under constraints**—for example, a consumer, firm, or gov

In [9]:
response.output_text

'“Agent” can mean different things depending on the field, but in the most general sense:\n\n**An agent is something that acts on behalf of a goal, and whose actions are responsive to its environment.**\n\nA good, broad definition usually includes these pieces:\n\n1. **It perceives or observes** something about the world.\n2. **It chooses actions** based on what it perceives.\n3. **Its actions affect outcomes** in the world.\n4. **It is organized around some objective, preference, or function**—even if only implicitly.\n\n### In philosophy and everyday language\nAn agent is a being that can **intend, decide, and act**.  \nA person is a paradigm agent because they can deliberate and choose among alternatives.\n\n### In economics and social science\nAn agent is an entity that **makes decisions under constraints**—for example, a consumer, firm, or government. The focus is on **behavior and incentives**.\n\n### In AI and computer science\nAn agent is usually a system that:\n- receives inpu

Agora a pergunta terá relação com a funcionalidade da ferramenta, então na primeira chamada o modelo aprende o `args` que deve ser usado numa eventual chamada da ferramenta, mas a chamada em si só ocorre na segunda interação com a api.

- **Primeira chamada: o modelo decide se usa ou não uma tool**

In [10]:
instructions = f"You are helping analyze a PubMed literature review. The current search strategy's query is: {PUBMED_QUERY}"


history = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": "Which pmids returns from the query in effect for this pipeline?"}
]


response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=history,
    tools=[search_pubmed_tool]  # SCHEMA/JSON definition
)

for item in response.output:
    print(item.type, item)

function_call ResponseFunctionToolCall(arguments='{"query":"( \\"Temporomandibular Joint\\"[Mesh] OR \\"temporomandibular joint\\"[Title/Abstract] ) AND ( \\"Joint Prosthesis\\"[Mesh] OR prosthe*[Title/Abstract] OR replacement[Title/Abstract] OR alloplastic[Title/Abstract] ) AND ( infection*[Title/Abstract] OR antibiotic*[Title/Abstract] OR prophylaxis[Title/Abstract] ) AND ( \\"2000/01/01\\"[Date - Publication] : \\"2025/12/31\\"[Date - Publication] )","limit":20}', call_id='call_37prDti1gxPVn9zKn7ki3Cur', name='search_pubmed', type='function_call', id='fc_06ef0fe64765ebfe006ac92e93a6f887d2b52bcf5180e40158', async_=None, caller=None, namespace=None, status='completed')


- **Resultado da primeira chamada: `arguments`**

Não há uma regra fixa, o modelo pode retornar em `arguments` a pergunta do usuário reformulada ou simplesmente copiar um query literal contido em `instructions` - nesse caso temos o `PUBMED_QUERY`.

In [11]:
call = response.output[0]
args = json.loads(call.arguments)

results = search_pubmed(**args)
result_json = json.dumps(results, indent=2)

result_json

'{\n  "header": {\n    "type": "esearch",\n    "version": "0.3"\n  },\n  "esearchresult": {\n    "count": "100",\n    "retmax": "20",\n    "retstart": "0",\n    "idlist": [\n      "41328123",\n      "41076314",\n      "40725551",\n      "40393895",\n      "40335417",\n      "40153056",\n      "40090550",\n      "39756382",\n      "39490003",\n      "39487073",\n      "39344460",\n      "39344453",\n      "39299812",\n      "39256068",\n      "39232864",\n      "38743036",\n      "38685147",\n      "38071499",\n      "37847424",\n      "37246472"\n    ],\n    "translationset": [],\n    "querytranslation": "(\\"Temporomandibular Joint\\"[MeSH Terms] OR \\"Temporomandibular Joint\\"[Title/Abstract]) AND (\\"Joint Prosthesis\\"[MeSH Terms] OR \\"prosthe*\\"[Title/Abstract] OR \\"replacement\\"[Title/Abstract] OR \\"alloplastic\\"[Title/Abstract]) AND (\\"infection*\\"[Title/Abstract] OR \\"antibiotic*\\"[Title/Abstract] OR \\"prophylaxis\\"[Title/Abstract]) AND 2000/01/01:2025/12/31[Date -

O objeto `function_call` inteiro (o `ResponseFunctionToolCall`) é o que entra em history, não só o texto ou os argumentos - é justamente esse objeto completo (com `.call_id`, `.name`, `.arguments`) que a API depois precisa ver de volta no histórico pra saber a quê o `function_call_output` que você manda em seguida está respondendo.

In [12]:
history += response.output

history.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})


response = openai_client.responses.create(
    model=MODEL,
    input=history,
    tools=[search_pubmed_tool],
)

response.output_text

'The query returns **100 PubMed records** total.\n\nThe first 20 PMIDs returned are:\n\n1. 41328123  \n2. 41076314  \n3. 40725551  \n4. 40393895  \n5. 40335417  \n6. 40153056  \n7. 40090550  \n8. 39756382  \n9. 39490003  \n10. 39487073  \n11. 39344460  \n12. 39344453  \n13. 39299812  \n14. 39256068  \n15. 39232864  \n16. 38743036  \n17. 38685147  \n18. 38071499  \n19. 37847424  \n20. 37246472\n\nIf you want, I can also help retrieve the next batches of PMIDs beyond the first 20.'

## **Agentic loop**

In [13]:
def make_call(call):
    # JSON string -> python dict
    args = json.loads(call.arguments)

    if call.name == "search_pubmed":
        result = search_pubmed(**args)

    # Turn results into JSON
    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [14]:
MAX_TOOL_CALLS = 2

instructions = f"You are helping analyze a PubMed literature review. The current search strategy's query is: {PUBMED_QUERY}"


def run_agent(question: str) -> dict:
    history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question},
    ]

    call_log = []
    stop_reason = None
    tool_call_count = 0
    final_answer = None
    it = 0

    while True:
        print(f'iteration #{it}...')
        has_function_calls = False

        response = openai_client.responses.create(
            model=MODEL,
            input=history,
            tools=[search_pubmed_tool],
        )

        history.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                if tool_call_count >= MAX_TOOL_CALLS:
                    stop_reason = 'max_tool_calls'
                    break

                print('function_call:', item.name, item.arguments)
                call_output = make_call(item)
                tool_call_count += 1

                call_log.append({
                    'tool': item.name,
                    'arguments': json.loads(item.arguments),
                    'result_summary': call_output['output'][:200],
                })

                history.append(call_output)
                has_function_calls = True

            elif item.type == 'message':
                final_answer = item.content[0].text
                print('ASSISTANT:')
                print(final_answer)

        if stop_reason == 'max_tool_calls':
            print(f'Parando: limite de {MAX_TOOL_CALLS} chamadas de tool atingido.')
            break

        it += 1
        if not has_function_calls:
            stop_reason = 'end_turn'
            break

    return {
        'answer': final_answer,
        'call_log': call_log,
        'stop_reason': stop_reason,
    }

## **Testando com três perguntas conhecidas**

A terceira pergunta não tem relação nenhuma com PubMed — serve pra confirmar que o agente também sabe **não** chamar a tool quando ela não é necessária.

In [15]:
known_questions = [
    "How many PMIDs does the query in effect for this pipeline return?",
    "What is the first PMID returned by our current search?",
    "What is the capital of France?",
]

for question in known_questions:
    print("=" * 60)
    print("QUESTION:", question)
    result = run_agent(question)
    print("STOP REASON:", result["stop_reason"])
    print("CALL LOG:", result["call_log"])
    print("ANSWER:", result["answer"])
    print()

QUESTION: How many PMIDs does the query in effect for this pipeline return?
iteration #0...
function_call: search_pubmed {"query":"( \"Temporomandibular Joint\"[Mesh] OR \"temporomandibular joint\"[Title/Abstract] ) AND ( \"Joint Prosthesis\"[Mesh] OR prosthe*[Title/Abstract] OR replacement[Title/Abstract] OR alloplastic[Title/Abstract] ) AND ( infection*[Title/Abstract] OR antibiotic*[Title/Abstract] OR prophylaxis[Title/Abstract] ) AND ( \"2000/01/01\"[Date - Publication] : \"2025/12/31\"[Date - Publication] )","limit":20}
iteration #1...
ASSISTANT:
The query returns **100 PMIDs** in total.
STOP REASON: end_turn
CALL LOG: [{'tool': 'search_pubmed', 'arguments': {'query': '( "Temporomandibular Joint"[Mesh] OR "temporomandibular joint"[Title/Abstract] ) AND ( "Joint Prosthesis"[Mesh] OR prosthe*[Title/Abstract] OR replacement[Title/Abstract] OR alloplastic[Title/Abstract] ) AND ( infection*[Title/Abstract] OR antibiotic*[Title/Abstract] OR prophylaxis[Title/Abstract] ) AND ( "2000/01/0